## this is complete Transformer component used in GPT-style models
### took help from chat GPT

### we will combine:
Causal Multi-Head Attention, 
Residual Connection, 
Layer Normalization, 
Feed-Forward Network (MLP), 
Another Residual Connection, 
Another Layer Normalization

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# Causal Self-Attention Head
class Head(nn.Module):

    def __init__(self, embed_dim, head_size):

        super().__init__()

        self.query = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

        self.key = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

    def forward(self, x):

        B, T, C = x.shape

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2, -1)

        scores = scores / (K.shape[-1] ** 0.5)

        mask = torch.tril(
            torch.ones(
                T,
                T,
                device=x.device
            )
        )

        scores = scores.masked_fill(
            mask == 0,
            float('-inf')
        )

        weights = F.softmax(
            scores,
            dim=-1
        )

        out = weights @ V

        return out

In [4]:
# Multi Head Attention
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads
    ):

        super().__init__()

        head_size = embed_dim // num_heads

        self.heads = nn.ModuleList(
            [
                Head(
                    embed_dim,
                    head_size
                )
                for _ in range(num_heads)
            ]
        )

        self.proj = nn.Linear(
            embed_dim,
            embed_dim
        )

    def forward(self, x):

        out = torch.cat(
            [
                h(x)
                for h in self.heads
            ],
            dim=-1
        )

        out = self.proj(out)

        return out

In [5]:
# Feed Forward Network
class FeedForward(nn.Module):

    def __init__(self, embed_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                embed_dim,
                4 * embed_dim
            ),

            nn.ReLU(),

            nn.Linear(
                4 * embed_dim,
                embed_dim
            )

        )

    def forward(self, x):

        return self.net(x)

In [7]:
# Transformer Block
class TransformerBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads
    ):

        super().__init__()

        self.sa = MultiHeadAttention(
            embed_dim,
            num_heads
        )

        self.ffwd = FeedForward(
            embed_dim
        )

        self.ln1 = nn.LayerNorm(
            embed_dim
        )

        self.ln2 = nn.LayerNorm(
            embed_dim
        )

    def forward(self, x):

        x = x + self.sa(
            self.ln1(x)
        )

        x = x + self.ffwd(
            self.ln2(x)
        )

        return x

In [8]:
# test 
B = 2
T = 10
C = 256

x = torch.randn(
    B,
    T,
    C
)

block = TransformerBlock(
    embed_dim=256,
    num_heads=8
)

out = block(x)

print(out.shape)

torch.Size([2, 10, 256])


In [9]:
total_params = sum(
    p.numel()
    for p in block.parameters()
)

print(
    f"{total_params:,}"
)

788,992
